# 🟠 Day 8 — Human-in-the-Loop & Time Travel

**LangGraph from Zero to Advanced · Day 8 of 16**

**Today's goal:** **pause** a graph, ask a human, and **continue** with their answer. Then go **back in time** to an earlier step and try something different.

Real uses: approve an email before it's sent, approve a payment, fix an AI draft, ask for missing info.

| Lesson | Topic |
|---|---|
| 1 | `interrupt()` — pause and ask |
| 2 | Approve or reject an action |
| 3 | Human edits the AI's draft |
| 4 | Approve tool calls in an agent |
| 5 | Ask again until the answer is valid |
| 6 | Change the state by hand: `update_state` |
| 7 | Time travel: replay & fork |
| 8 | Practice |

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Lesson 1 — `interrupt()`: pause and ask

Inside any node, call `interrupt(something)`:

1. The graph **pauses** and saves its state (so a **checkpointer is required**).
2. `invoke()` returns, with the question under the key **`__interrupt__`**.
3. Later you **resume** with `invoke(Command(resume=answer), config)` — the same thread.
4. `interrupt(...)` then **returns the answer** inside the node, and the node continues.

No LLM in this first example — just the mechanics.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class GreetState(TypedDict):
    name: str
    greeting: str

def ask_name(state: GreetState) -> dict:
    name = interrupt("What is your name?")      # ⏸ pauses here
    return {"name": name}                       # ▶ runs after resume

def greet(state: GreetState) -> dict:
    return {"greeting": f"Namaste, {state['name']}! 🙏"}

g = StateGraph(GreetState)
g.add_sequence([ask_name, greet])
g.add_edge(START, "ask_name")
g.add_edge("greet", END)
greet_app = g.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "greet-1"}}

# Run until the pause
result = greet_app.invoke({}, config)
print("Paused! Question:", result["__interrupt__"][0].value)
print("Next node:", greet_app.get_state(config).next)

In [ ]:
# The human answers → resume
result = greet_app.invoke(Command(resume="Sneha"), config)
print(result["greeting"])
assert result["greeting"] == "Namaste, Sneha! 🙏"
print("✅ Paused and resumed!")

⚠️ **Important rule:** when you resume, the paused node runs **again from its first line**. `interrupt()` then returns the answer instead of pausing. So don't put side effects (sending emails, payments) **before** `interrupt()` in the same node.

## Lesson 2 — Approve or reject an action

The AI writes an email. A human must approve it before it's "sent". We use `Command(goto=...)` (Day 3) to choose the next node from the answer.

```
START ► write_email ► human_approval ─┬─ yes ► send_email ► END
                                      └─ no ──► cancel ────► END
```

In [ ]:
from typing import Literal

class EmailState(TypedDict):
    request: str
    draft: str
    status: str

def write_email(state: EmailState) -> dict:
    draft = llm.invoke(f"Write a short, polite email (max 60 words) for: {state['request']}").content
    return {"draft": draft}

def human_approval(state: EmailState) -> Command[Literal["send_email", "cancel"]]:
    answer = interrupt({"question": "Send this email? (yes/no)", "draft": state["draft"]})
    if answer.strip().lower() in ("yes", "y"):
        return Command(goto="send_email")
    return Command(goto="cancel")

def send_email(state: EmailState) -> dict:
    print("📧 (pretend) sending email...")
    return {"status": "sent ✅"}

def cancel(state: EmailState) -> dict:
    return {"status": "cancelled ❌"}

g = StateGraph(EmailState)
for fn in (write_email, human_approval, send_email, cancel):
    g.add_node(fn.__name__, fn)
g.add_edge(START, "write_email")
g.add_edge("write_email", "human_approval")
g.add_edge("send_email", END)
g.add_edge("cancel", END)
email_app = g.compile(checkpointer=InMemorySaver())
show_graph(email_app)

In [ ]:
cfg = {"configurable": {"thread_id": "email-1"}}
paused = email_app.invoke({"request": "ask my manager for leave on Friday"}, cfg)

question = paused["__interrupt__"][0].value
print("❓", question["question"])
print("📝 Draft:\n", question["draft"])

In [ ]:
# Try "no" as well, with a new thread_id!
final = email_app.invoke(Command(resume="yes"), cfg)
print("Status:", final["status"])

## Lesson 3 — Human edits the AI's draft

Instead of yes/no, the human can **return new text**. Whatever you pass to `Command(resume=...)` is what `interrupt()` returns.

In [ ]:
class PostState(TypedDict):
    topic: str
    post: str

def draft_post(state: PostState) -> dict:
    return {"post": llm.invoke(f"Write a 2-line LinkedIn post about {state['topic']}.").content}

def human_edit(state: PostState) -> dict:
    edited = interrupt({"instruction": "Edit the post (or send it back unchanged)", "post": state["post"]})
    return {"post": edited}

def publish(state: PostState) -> dict:
    print("🚀 Published:\n", state["post"])
    return {}

g = StateGraph(PostState)
g.add_sequence([draft_post, human_edit, publish])
g.add_edge(START, "draft_post")
g.add_edge("publish", END)
post_app = g.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "post-1"}}
paused = post_app.invoke({"topic": "finishing a LangGraph course"}, cfg)
print("AI draft:\n", paused["__interrupt__"][0].value["post"])

In [ ]:
my_version = "Just finished Day 8 of my LangGraph course — my agents now ask before they act! 🤖✋ #LangGraph"
result = post_app.invoke(Command(resume=my_version), cfg)
assert result["post"] == my_version

## Lesson 4 — Approve tool calls in an agent

For risky tools (payments, deleting data), call `interrupt()` **inside the tool**. The agent pauses right before the action.

In [ ]:
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

@tool
def transfer_money(to: str, amount: float) -> str:
    """Transfer money (in rupees) to a person."""
    decision = interrupt(f"Approve transfer of ₹{amount} to {to}? (yes/no)")
    if decision.lower() != "yes":
        return "Transfer cancelled by the user."
    return f"✅ Transferred ₹{amount} to {to}."

bank_tools = [transfer_money]
bank_llm = llm.bind_tools(bank_tools)

def bank_agent(state: MessagesState) -> dict:
    return {"messages": [bank_llm.invoke(state["messages"])]}

g = StateGraph(MessagesState)
g.add_node("agent", bank_agent)
g.add_node("tools", ToolNode(bank_tools))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", tools_condition)
g.add_edge("tools", "agent")
bank_app = g.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "bank-1"}}
paused = bank_app.invoke({"messages": [("user", "Send 500 rupees to Kiran")]}, cfg)
print("⏸", paused["__interrupt__"][0].value)

In [ ]:
done = bank_app.invoke(Command(resume="yes"), cfg)
print("🤖", done["messages"][-1].content)

## Lesson 5 — Ask again until the answer is valid

Put `interrupt()` in a `while` loop inside the node. Each resume gives a new answer; the node keeps asking until it's valid.

In [ ]:
class AgeState(TypedDict):
    age: int

def ask_age(state: AgeState) -> dict:
    question = "How old are you?"
    while True:
        answer = interrupt(question)
        if str(answer).isdigit() and 0 < int(answer) < 120:
            return {"age": int(answer)}
        question = f"'{answer}' is not a valid age. Please enter a number:"

g = StateGraph(AgeState)
g.add_node("ask_age", ask_age)
g.add_edge(START, "ask_age")
g.add_edge("ask_age", END)
age_app = g.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "age-1"}}
r = age_app.invoke({}, cfg)
for reply in ["twenty", "-5", "29"]:
    print("❓", r["__interrupt__"][0].value, "→ human says:", reply)
    r = age_app.invoke(Command(resume=reply), cfg)
print("Final:", r)
assert r == {"age": 29}

## Lesson 6 — Change the state by hand: `update_state`

You (or an admin UI) can **edit** a thread's saved state. The update goes through the reducers, just as if a node had returned it.

In [ ]:
cfg = {"configurable": {"thread_id": "greet-2"}}
greet_app.invoke({}, cfg)                                  # pauses at ask_name
greet_app.update_state(cfg, {"name": "Admin-set name"})    # edit the saved state
print(greet_app.get_state(cfg).values)

💡 **Static breakpoints (for debugging):** `compile(checkpointer=..., interrupt_before=["node"])` pauses before a node *without* changing the node's code. Resume with `invoke(None, config)`.

In [ ]:
debug_app = StateGraph(GreetState)
debug_app.add_node("greet", greet)
debug_app.add_edge(START, "greet")
debug_app.add_edge("greet", END)
debug_app = debug_app.compile(checkpointer=InMemorySaver(), interrupt_before=["greet"])

cfg = {"configurable": {"thread_id": "debug-1"}}
debug_app.invoke({"name": "Vikram"}, cfg)
print("Paused before:", debug_app.get_state(cfg).next)
print(debug_app.invoke(None, cfg))      # None = just continue

## Lesson 7 — Time travel: replay & fork

Because every step is saved (Day 7), you can go back to **any** checkpoint:

* **Replay:** `invoke(None, old_checkpoint_config)` → re-runs from that point.
* **Fork:** `update_state(old_checkpoint_config, {...})` → creates a **new branch** with changed values, then `invoke(None, new_config)`.

Example: a joke generator. We go back and change the topic — without starting over.

In [ ]:
class JokeState(TypedDict):
    topic: str
    joke: str

def write_joke(state: JokeState) -> dict:
    return {"joke": llm.invoke(f"Tell a one-line clean joke about {state['topic']}.").content}

g = StateGraph(JokeState)
g.add_node("write_joke", write_joke)
g.add_edge(START, "write_joke")
g.add_edge("write_joke", END)
joke_app = g.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "jokes"}}
print("Original:", joke_app.invoke({"topic": "programmers"}, cfg)["joke"])

In [ ]:
# 1) find the checkpoint just BEFORE write_joke ran
history = list(joke_app.get_state_history(cfg))
for s in history:
    print("step", s.metadata["step"], "| next:", s.next, "| topic:", s.values.get("topic"))

before_joke = next(s for s in history if s.next == ("write_joke",))

In [ ]:
# 2) FORK: change the topic at that checkpoint, then continue from there
fork_config = joke_app.update_state(before_joke.config, {"topic": "cats"})
print("Forked:", joke_app.invoke(None, fork_config)["joke"])

# The thread now has both versions in its history
print("Checkpoints in thread:", len(list(joke_app.get_state_history(cfg))))

## Lesson 8 — Practice

**Exercise 1.** Build `START → plan_trip (LLM) → human_review → END`. In `human_review`, the human answers either `"ok"` (keep the plan) or a new instruction; if it's an instruction, store it in `feedback`.

**Exercise 2.** Build a graph that asks for two numbers using two `interrupt()` calls in **two different nodes**, then adds them. Resume twice.

In [ ]:
# ✅ Solution 1
class TripState(TypedDict):
    city: str
    plan: str
    feedback: str

def plan_trip(state):
    return {"plan": llm.invoke(f"Give a 3-item one-day plan for {state['city']}. Very short.").content}

def human_review(state):
    answer = interrupt({"plan": state["plan"], "ask": "Reply 'ok' or give feedback"})
    return {} if answer == "ok" else {"feedback": answer}

g = StateGraph(TripState)
g.add_sequence([plan_trip, human_review])
g.add_edge(START, "plan_trip")
g.add_edge("human_review", END)
trip_app = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "trip"}}
trip_app.invoke({"city": "Jaipur"}, cfg)
print(trip_app.invoke(Command(resume="Add a food stop"), cfg))

In [ ]:
# ✅ Solution 2
class SumState(TypedDict):
    a: int
    b: int
    total: int

def ask_a(s): return {"a": int(interrupt("First number?"))}
def ask_b(s): return {"b": int(interrupt("Second number?"))}
def add(s):   return {"total": s["a"] + s["b"]}

g = StateGraph(SumState)
g.add_sequence([ask_a, ask_b, add])
g.add_edge(START, "ask_a")
g.add_edge("add", END)
sum_app = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "sum"}}
sum_app.invoke({}, cfg)
sum_app.invoke(Command(resume="7"), cfg)
res = sum_app.invoke(Command(resume="35"), cfg)
print(res)
assert res["total"] == 42
print("✅ All exercises solved!")

## 🎯 Day 8 Recap

* `interrupt(value)` pauses; `invoke(Command(resume=answer), config)` continues. Needs a checkpointer.
* The paused node **re-runs from the top** on resume.
* Use it for approvals, edits, tool approval, and input validation.
* `update_state` edits saved state; `interrupt_before` is a debug breakpoint.
* **Time travel:** pick a checkpoint from `get_state_history` → replay or fork.

**Tomorrow (Day 9):** **streaming** — show tokens and progress live, plus async.